# Crop and Weed Detection with YOLO

This notebook downloads the public crop/weed dataset, validates its YOLO labels, creates train/validation/test splits, trains a lightweight YOLO model, evaluates it, and exports it to ONNX for the Flask website.

Before running: choose **Runtime > Change runtime type > T4 GPU**.

In [ ]:
!pip install -q ultralytics kagglehub onnx onnxslim

In [ ]:
from pathlib import Path
import random
import shutil
import torch
import yaml
import kagglehub

print('GPU available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('Warning: training will be slow. Select a T4 GPU from the Runtime menu.')

## Save checkpoints safely in Google Drive

Training checkpoints are written to Google Drive after every epoch, so they are not lost if the Colab runtime disconnects.

In [ ]:
from pathlib import Path
from google.colab import drive
drive.mount('/content/drive')

drive_project = Path('/content/drive/MyDrive/Smart_Agriculture_YOLO')
drive_project.mkdir(parents=True, exist_ok=True)
print('Persistent project folder:', drive_project)

## 1. Download the dataset

Source: https://www.kaggle.com/datasets/ravirajsinh45/crop-and-weed-detection-data-with-bounding-boxes

The dataset contains 1,300 images and two classes: `crop` and `weed`.

In [ ]:
dataset_path = Path(kagglehub.dataset_download(
    'ravirajsinh45/crop-and-weed-detection-data-with-bounding-boxes'
))
print('Dataset path:', dataset_path)

## 2. Validate image-label pairs and create the splits

In [ ]:
IMAGE_EXTENSIONS = {'.jpg', '.jpeg', '.png'}
EXCLUDED_LABEL_FILES = {'classes.txt', 'obj.names', 'train.txt', 'test.txt'}

def is_valid_yolo_label(label_path):
    lines = [line.strip() for line in label_path.read_text().splitlines() if line.strip()]
    if not lines:
        return False
    for line in lines:
        values = line.split()
        if len(values) != 5:
            return False
        class_id = int(float(values[0]))
        coordinates = [float(value) for value in values[1:]]
        if class_id not in (0, 1) or not all(0 <= value <= 1 for value in coordinates):
            return False
    return True

label_files = {}
for label_path in dataset_path.rglob('*.txt'):
    if label_path.name.lower() in EXCLUDED_LABEL_FILES:
        continue
    try:
        if is_valid_yolo_label(label_path):
            label_files.setdefault(label_path.stem, label_path)
    except (ValueError, UnicodeDecodeError):
        continue

pairs = []
for image_path in dataset_path.rglob('*'):
    if image_path.suffix.lower() in IMAGE_EXTENSIONS and image_path.stem in label_files:
        pairs.append((image_path, label_files[image_path.stem]))

assert len(pairs) >= 1000, f'Expected about 1300 labeled images, found only {len(pairs)}'
print('Valid labeled images:', len(pairs))

In [ ]:
output_root = Path('/content/crop_weed_yolo')
if output_root.exists():
    shutil.rmtree(output_root)

random.seed(42)
random.shuffle(pairs)
train_end = int(len(pairs) * 0.70)
val_end = train_end + int(len(pairs) * 0.20)
splits = {
    'train': pairs[:train_end],
    'val': pairs[train_end:val_end],
    'test': pairs[val_end:],
}

for split_name, split_pairs in splits.items():
    image_dir = output_root / 'images' / split_name
    label_dir = output_root / 'labels' / split_name
    image_dir.mkdir(parents=True, exist_ok=True)
    label_dir.mkdir(parents=True, exist_ok=True)
    for number, (image_path, label_path) in enumerate(split_pairs):
        unique_stem = f'{number:04d}_{image_path.stem}'
        shutil.copy2(image_path, image_dir / f'{unique_stem}{image_path.suffix.lower()}')
        shutil.copy2(label_path, label_dir / f'{unique_stem}.txt')
    print(split_name, len(split_pairs))

data_yaml = {
    'path': str(output_root),
    'train': 'images/train',
    'val': 'images/val',
    'test': 'images/test',
    'names': {0: 'crop', 1: 'weed'},
}
data_yaml_path = output_root / 'data.yaml'
data_yaml_path.write_text(yaml.safe_dump(data_yaml, sort_keys=False))
print(data_yaml_path.read_text())

In [ ]:
box_counts = {0: 0, 1: 0}
for split_name in splits:
    for label_path in (output_root / 'labels' / split_name).glob('*.txt'):
        for line in label_path.read_text().splitlines():
            if line.strip():
                box_counts[int(float(line.split()[0]))] += 1
print('Crop boxes:', box_counts[0])
print('Weed boxes:', box_counts[1])

## 3. Train the lightweight YOLO model

The pretrained nano model is fine-tuned for the two project classes. Early stopping is enabled to avoid unnecessary training.

In [ ]:
import importlib.util
import subprocess
import sys

if importlib.util.find_spec('ultralytics') is None:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'ultralytics', 'onnx', 'onnxslim'])

from ultralytics import YOLO

model = YOLO('yolo11n.pt')
model.train(
    data=str(data_yaml_path),
    epochs=30,
    imgsz=512,
    batch=16,
    patience=8,
    seed=42,
    device=0 if torch.cuda.is_available() else 'cpu',
    project=str(drive_project),
    name='crop_weed_yolo11n',
    exist_ok=True,
)

## 4. Evaluate on the unseen test split

In [ ]:
run_dir = drive_project / 'crop_weed_yolo11n'
best_weights = run_dir / 'weights' / 'best.pt'
best_model = YOLO(str(best_weights))
metrics = best_model.val(data=str(data_yaml_path), split='test', imgsz=512)

print(f'Precision: {metrics.box.mp:.4f}')
print(f'Recall: {metrics.box.mr:.4f}')
print(f'mAP50: {metrics.box.map50:.4f}')
print(f'mAP50-95: {metrics.box.map:.4f}')

## 5. Test one image visually

In [ ]:
sample_image = next((output_root / 'images' / 'test').iterdir())
prediction = best_model.predict(source=str(sample_image), conf=0.25, save=True)
prediction[0].show()

## 6. Export ONNX and download the deliverables

The website will use the ONNX file without installing PyTorch or Ultralytics on Render.

In [ ]:
exported_path = Path(best_model.export(
    format='onnx',
    imgsz=512,
    simplify=True,
    dynamic=False,
    nms=True,
))
final_onnx = drive_project / 'weed_detector.onnx'
shutil.copy2(exported_path, final_onnx)
print('ONNX model:', final_onnx, final_onnx.stat().st_size / (1024 * 1024), 'MB')

In [ ]:
results_archive = shutil.make_archive(str(drive_project / 'weed_training_results'), 'zip', run_dir)
print('Results archive:', results_archive)

from google.colab import files
files.download(str(final_onnx))
files.download(results_archive)